In [1]:
# --- Step 1: Import Libraries ---
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported.")

Libraries imported.


In [2]:
# --- Step 2: Load the PaySim Dataset ---
try:
    df = pd.read_csv('Paysim.csv')
    print("PaySim dataset loaded successfully.")
except FileNotFoundError:
    print("FATAL ERROR: 'Paysim.csv' not found.")
    print("Please download the dataset from Kaggle, rename it, and place it in your 'system' folder.")
    # Stop execution if file not found
    raise



PaySim dataset loaded successfully.


In [3]:
# --- Step 2: Load the PaySim Dataset ---
print("Loading PaySim dataset (Paysim.csv)... This may take a few minutes.")
try:
    df = pd.read_csv('Paysim.csv')
    print("Dataset loaded successfully.")
    print(f"Original dataset size: {len(df)} rows")
except FileNotFoundError:
    print("="*50)
    print("ERROR: 'Paysim.csv' not found. Please download it from Kaggle and place it in your 'system' folder.")
    print("="*50)
    # Stop execution if file not found
    raise SystemExit("File not found.")

print("-" * 50)

Loading PaySim dataset (Paysim.csv)... This may take a few minutes.


Dataset loaded successfully.
Original dataset size: 6362620 rows
--------------------------------------------------


In [4]:
# --- Step 3: Sub-Sampling for Speed ---
df_sample = df.sample(n=500000, random_state=42)
print(f"Dataset sub-sampled to 500,000 rows.")


Dataset sub-sampled to 500,000 rows.


In [5]:
# --- Step 4: Feature Engineering (Features 1 & 2) ---
df_sample['amount_gt_oldbalanceOrg'] = (df_sample['amount'] > df_sample['oldbalanceOrg']).astype(int)
df_sample['newbalanceOrig_is_zero'] = (df_sample['newbalanceOrig'] == 0).astype(int)
df_sample['oldbalanceDest_is_zero'] = (df_sample['oldbalanceDest'] == 0).astype(int)
df_sample = pd.get_dummies(df_sample, columns=['type'], drop_first=False, dtype=int)
print("Feature engineering complete.")

Feature engineering complete.


In [6]:
# --- Step 5: Data Preparation (THE CRITICAL FIX) ---

# 5.1: Define ALL features the model will use
# This list MUST match the one in app.py
ALL_FEATURES = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest',
    'amount_gt_oldbalanceOrg', 'newbalanceOrig_is_zero', 'oldbalanceDest_is_zero',
    'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER'
]



In [7]:
# 5.2: Define which features are numerical (to be scaled)
NUMERICAL_FEATURES = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest'
]

In [8]:
# 5.3: Define which features are binary (to be left alone)
BINARY_FEATURES = [
    'amount_gt_oldbalanceOrg', 'newbalanceOrig_is_zero', 'oldbalanceDest_is_zero',
    'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER'
]

In [9]:
# 5.4: Prepare X and y
y = df_sample['isFraud']
X = df_sample[ALL_FEATURES] # Use only the features we defined
print("X and y created.")


X and y created.


In [10]:
# 5.5: Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print("Data split into train and test sets.")

Data split into train and test sets.


In [11]:
# 5.6: Correctly scale ONLY the numerical features
scaler = StandardScaler()
X_train[NUMERICAL_FEATURES] = scaler.fit_transform(X_train[NUMERICAL_FEATURES])
X_test[NUMERICAL_FEATURES] = scaler.transform(X_test[NUMERICAL_FEATURES])
print("Numerical features scaled correctly.")

Numerical features scaled correctly.


In [12]:
# --- Step 6: Train the New "P2P Expert" Model ---
print("\nTraining new 'P2P Expert' (Random Forest) model...")
print("This may take a few minutes...")


Training new 'P2P Expert' (Random Forest) model...
This may take a few minutes...


In [13]:
# We use n_jobs=1 to prevent the server deadlock
rf_model_paysim = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1) 
rf_model_paysim.fit(X_train, y_train)

print("Model B (PaySim) training complete!")
print("-" * 50)

Model B (PaySim) training complete!
--------------------------------------------------


In [14]:
# --- Step 7: Evaluate the New Model ---
y_pred_paysim = rf_model_paysim.predict(X_test)
print(classification_report(y_test, y_pred_paysim))
print("-" * 50)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    149806
           1       1.00      1.00      1.00       194

    accuracy                           1.00    150000
   macro avg       1.00      1.00      1.00    150000
weighted avg       1.00      1.00      1.00    150000

--------------------------------------------------


In [15]:
# --- Step 8: Save the New "Expert" Model and Scaler ---
joblib.dump(rf_model_paysim, 'model_paysim.joblib')
joblib.dump(scaler, 'paysim_scaler.joblib')
print("SUCCESS: 'model_paysim.joblib' and 'paysim_scaler.joblib' have been saved!")
print("You are now ready to run your server.")

SUCCESS: 'model_paysim.joblib' and 'paysim_scaler.joblib' have been saved!
You are now ready to run your server.
